# Group 03 (Team ElvishBhai): Image Classification Challenge, training notebook

**Final model.** Ensemble of 3 ResNet-18 networks (logits averaged), initialised from the standard ImageNet weights `torchvision ResNet18_Weights.IMAGENET1K_V1` and fine-tuned only on the kit data (train + valid, 15,729 images). No other pretrained weights and no external data. The architecture is re-implemented in Cell 5 with torchvision parameter names, so `predict.py` needs only `numpy` and `torch`.

**Recipe** (`CFG['models']['r18_in_lr03']`, used by Cell 9): grayscale 28×28 upsampled to 112×112 (bilinear) and copied to 3 channels, ImageNet mean/std normalisation; random shift ±2 px (no flips); SGD with Nesterov momentum 0.9, lr 0.03, weight decay 5e-4 (none on BatchNorm and bias), batch 128, per-step cosine schedule, 15 epochs, bf16 autocast; seeds 0, 1, 2; weights from the final epoch.

**Selection protocol.** Candidates were trained on the train split only and scored on the provided valid split at the final epoch, 3 seeds each. Adoption rule fixed in advance: macro-F1 gain of at least +1.0 point, every seed positive and paired-bootstrap 95% CI above 0 (per seed or for the 3-seed ensemble); accuracy, then model size, break ties.

**Estimate** (train-only models; valid was used for selection, so optimistic; the README says the test set is harder): macro-F1 0.8818, accuracy 0.9299 per model; 3-model ensemble macro-F1 0.8818, accuracy 0.9329. The main remaining error is the confusion between classes 5 and 7, which a pixel-level check found nearly indistinguishable.

**Experiments not kept in this notebook** (experiment runner and comparison cells removed; full record in the team's experiment log). Adopted: ImageNet ResNet-18 over a from-scratch ResNet (0.31M parameters, macro-F1 0.8639), then lr 0.03 over lr 0.01 (+1.28 points). Not adopted: rotation/scale/intensity augmentation, no shift, 224 px input, 30 epochs, lr 0.003/0.06/0.1, ResNet-34, ResNet-50, label smoothing, weight decay 1e-4, mixup, CutMix, logit adjustment, test-time shift averaging, mixed ResNet-18 + scratch ensemble. Some of these runs were made in a parallel session with the same code.

**Reproduce.** Run Cells 1–6, then Cell 9 (writes `member_0.pt`–`member_2.pt` to `CFG['sub_dir']`), then Cell 10 (checks `predict.py` on valid on GPU and CPU). The environment is recorded in Cell 3's output. Two separate final-fit runs printed identical training losses (0.0267, 0.0263, 0.0257). Cell 10's valid score is a wiring check only, because the final members were trained on valid.

In [1]:
# Cell 1: Imports
import gzip, os, struct, time, json, math, glob, random, platform, importlib.util
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from google.colab import drive
from tqdm.auto import tqdm

In [2]:
# Cell 2: Configuration
_R18 = dict(arch='r18', pretrained=True, size=112, epochs=15, batch=128, lr=0.01, wd=5e-4, ls=0.0, dropout=0.0,
            shift=2, rot=0.0, scale=0.0, bright=0.0, contrast=0.0)
_SMALL = dict(arch='small', pretrained=False, size=28, epochs=40, batch=128, lr=0.1, wd=5e-4, ls=0.0, dropout=0.2,
              shift=2, rot=0.0, scale=0.0, bright=0.0, contrast=0.0)
_AUG = dict(rot=10.0, scale=0.1, bright=0.15, contrast=0.2)
CFG = dict(
    data_dir='/content/drive/MyDrive/data',
    out_dir='/content/drive/MyDrive/models',
    sub_dir='/content/drive/MyDrive/models/Group 3',
    num_classes=11,
    train_counts=[1112, 767, 1920, 1493, 1082, 595, 1077, 604, 710, 688, 3327],
    valid_counts=[183, 264, 274, 202, 131, 90, 156, 100, 248, 237, 469],
    imagenet_mean=(0.485, 0.456, 0.406),
    imagenet_std=(0.229, 0.224, 0.225),
    momentum=0.9,
    amp=True,
    seeds=(0, 1, 2),
    la_tau=1.0,
    adopt_min_gain=0.01,
    bootstrap=2000,
    models=dict(
        r18_in=_R18,
        scratch=_SMALL,
        r18_in_aug={**_R18, **_AUG},
        scratch_aug={**_SMALL, **_AUG},
        scratch_noshift={**_SMALL, 'shift': 0},
        r18_in_noshift={**_R18, 'shift': 0},
        r18_in_224={**_R18, 'size': 224},
        r18_in_ep30={**_R18, 'epochs': 30},
        scratch_ep80={**_SMALL, 'epochs': 80},
        r18_in_lr03={**_R18, 'lr': 0.03},
    ),
    queue=['r18_in', 'scratch', 'scratch_aug', 'r18_in_aug',
           'scratch_noshift', 'r18_in_noshift', 'scratch_ep80', 'r18_in_ep30', 'r18_in_224'],
    comparisons=[('scratch', 'r18_in', False, False),
                 ('scratch', 'scratch', False, True),
                 ('scratch', 'scratch_noshift', False, False),
                 ('r18_in', 'r18_in_noshift', False, False),
                 ('scratch', 'scratch_ep80', False, False),
                 ('r18_in', 'r18_in_ep30', False, False),
                 ('r18_in', 'r18_in_224', False, False)],
    final_model='r18_in_lr03',
    final_la=False,
    final_fit='train+valid',
    final_seeds=(0, 1, 2),
)

In [3]:
# Cell 3: Drive mount, environment record and seeding
drive.mount('/content/drive')
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
assert torch.cuda.is_available(), 'GPU runtime required: Runtime -> Change runtime type -> GPU'
dev = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

os.makedirs(CFG['out_dir'], exist_ok=True)
os.makedirs(CFG['sub_dir'], exist_ok=True)
ENV = dict(python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__,
           numpy=np.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
           gpu=torch.cuda.get_device_name(0),
           gpu_mem_gb=round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
           cudnn_tf32=torch.backends.cudnn.allow_tf32, matmul_tf32=torch.backends.cuda.matmul.allow_tf32,
           deterministic=True, amp_bf16=CFG['amp'])
with open(os.path.join(CFG['out_dir'], 'env.json'), 'w') as f:
    json.dump(ENV, f, indent=1)
print(json.dumps(ENV, indent=1))

Mounted at /content/drive
{
 "python": "3.13.16",
 "torch": "2.11.0+cu130",
 "torchvision": "0.26.0+cu130",
 "numpy": "2.4.6",
 "cuda": "13.0",
 "cudnn": 92700,
 "gpu": "NVIDIA A100-SXM4-80GB",
 "gpu_mem_gb": 79.3,
 "cudnn_tf32": true,
 "matmul_tf32": false,
 "deterministic": true,
 "amp_bf16": true
}


In [4]:
# Cell 4: Data loading and integrity checks
def load_idx(path):
    with gzip.open(path, 'rb') as f:
        _, _, _, ndim = struct.unpack('>BBBB', f.read(4))
        shape = struct.unpack(f'>{ndim}I', f.read(4 * ndim))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(shape)

D = CFG['data_dir']
Xtr_np = load_idx(f'{D}/train-images-idx3-ubyte.gz')
Ytr_np = load_idx(f'{D}/train-labels-idx1-ubyte.gz').astype(np.int64)
Xva_np = load_idx(f'{D}/valid-images-idx3-ubyte.gz')
Yva_np = load_idx(f'{D}/valid-labels-idx1-ubyte.gz').astype(np.int64)
assert Xtr_np.shape == (13375, 28, 28) and Xva_np.shape == (2354, 28, 28)
assert np.bincount(Ytr_np, minlength=11).tolist() == CFG['train_counts']
assert np.bincount(Yva_np, minlength=11).tolist() == CFG['valid_counts']

def to_t(a):
    return torch.from_numpy(a.copy()).float().div(255).unsqueeze(1).to(dev)

Xtr, Xva = to_t(Xtr_np), to_t(Xva_np)
Ytr, Yva = torch.from_numpy(Ytr_np).to(dev), torch.from_numpy(Yva_np).to(dev)
PRIOR_TR = np.bincount(Ytr_np, minlength=11) / len(Ytr_np)
print('train', tuple(Xtr.shape), 'valid', tuple(Xva.shape), f'train mean {Xtr.mean().item():.4f} std {Xtr.std().item():.4f}')

train (13375, 1, 28, 28) valid (2354, 1, 28, 28) train mean 0.4953 std 0.2826


In [5]:
# Cell 5: Model definitions
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.downsample = None
        if stride != 1 or cin != cout:
            self.downsample = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return F.relu(y + (x if self.downsample is None else self.downsample(x)))


class ResNet18Body(nn.Module):
    out_dim = 512

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        for i, (a, b, s) in enumerate([(64, 64, 1), (64, 128, 2), (128, 256, 2), (256, 512, 2)], 1):
            setattr(self, f'layer{i}', nn.Sequential(Block(a, b, s), Block(b, b, 1)))

    def forward(self, x):
        x = self.maxpool(F.relu(self.bn1(self.conv1(x))))
        return self.layer4(self.layer3(self.layer2(self.layer1(x)))).mean((2, 3))


class SmallResNet(nn.Module):
    out_dim = 128

    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.blocks = nn.Sequential(Block(32, 32, 1), Block(32, 64, 2), Block(64, 128, 2))

    def forward(self, x):
        return self.blocks(self.stem(x)).mean((2, 3))


class Net(nn.Module):
    def __init__(self, arch, num_classes, size, dropout, mean=None, std=None):
        super().__init__()
        self.size = size
        self.body = ResNet18Body() if arch == 'r18' else SmallResNet()
        ch = 3 if arch == 'r18' else 1
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(self.body.out_dim, num_classes)
        self.register_buffer('mean', torch.tensor(mean if mean is not None else [0.0] * ch, dtype=torch.float32).view(1, ch, 1, 1))
        self.register_buffer('std', torch.tensor(std if std is not None else [1.0] * ch, dtype=torch.float32).view(1, ch, 1, 1))
        self.register_buffer('logit_bias', torch.zeros(num_classes))

    def forward(self, x):
        if x.shape[-1] != self.size:
            x = F.interpolate(x, size=(self.size, self.size), mode='bilinear', align_corners=False)
        x = (x.expand(-1, self.mean.shape[1], -1, -1) - self.mean) / self.std
        return self.fc(self.drop(self.body(x))) + self.logit_bias


def build_net(c, mean, std):
    net = Net(c['arch'], CFG['num_classes'], c['size'], c['dropout'], mean, std)
    if c['pretrained']:
        sd = {k: v for k, v in resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).state_dict().items() if not k.startswith('fc.')}
        net.body.load_state_dict(sd, strict=True)
    return net


for _name, _c in CFG['models'].items():
    _n = build_net({**_c, 'pretrained': False}, None, None)
    print(f"{_name:12s} params {sum(p.numel() for p in _n.parameters()):,}")
_n = build_net(CFG['models']['r18_in'], CFG['imagenet_mean'], CFG['imagenet_std'])
print('ImageNet weights loaded into ResNet18Body (strict)')
del _n

r18_in       params 11,182,155
scratch      params 308,203
r18_in_aug   params 11,182,155
scratch_aug  params 308,203
scratch_noshift params 308,203
r18_in_noshift params 11,182,155
r18_in_224   params 11,182,155
r18_in_ep30  params 11,182,155
scratch_ep80 params 308,203
r18_in_lr03  params 11,182,155
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 229MB/s]

ImageNet weights loaded into ResNet18Body (strict)


In [6]:
# Cell 6: Augmentation, training and evaluation functions
def augment(x, c, g):
    n = x.shape[0]

    def u():
        return torch.rand(n, device=x.device, generator=g) * 2 - 1

    ang = u() * math.radians(c['rot'])
    sc = 1 + u() * c['scale']
    tx, ty = u() * c['shift'] * 2 / 28, u() * c['shift'] * 2 / 28
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(theta, list(x.shape), align_corners=False),
                      mode='bilinear', padding_mode='border', align_corners=False)
    if c['bright'] or c['contrast']:
        m = x.mean((1, 2, 3), keepdim=True)
        x = ((x - m) * (1 + u().view(-1, 1, 1, 1) * c['contrast']) + m + u().view(-1, 1, 1, 1) * c['bright']).clamp(0, 1)
    return x


def param_groups(net, wd):
    dec = [p for p in net.parameters() if p.ndim > 1]
    nod = [p for p in net.parameters() if p.ndim <= 1]
    return [dict(params=dec, weight_decay=wd), dict(params=nod, weight_decay=0.0)]


def train_model(name, seed, X, Y):
    c = CFG['models'][name]
    set_seed(seed)
    if c['pretrained']:
        mean, std = CFG['imagenet_mean'], CFG['imagenet_std']
    else:
        mean, std = (X.mean().item(),), (X.std().item(),)
    net = build_net(c, mean, std).to(dev)
    opt = torch.optim.SGD(param_groups(net, c['wd']), lr=c['lr'], momentum=CFG['momentum'], nesterov=True)
    n, bs = len(X), c['batch']
    total = c['epochs'] * math.ceil(n / bs)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * min(s, total) / total)))
    g = torch.Generator(device=dev)
    g.manual_seed(seed)
    t0 = time.time()
    bar = tqdm(range(c['epochs']), desc=f'{name} s{seed}', leave=False)
    for ep in bar:
        net.train()
        perm = torch.randperm(n, device=dev, generator=g)
        tl = torch.zeros((), device=dev)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            xb = augment(X[idx], c, g)
            with torch.autocast('cuda', dtype=torch.bfloat16, enabled=CFG['amp']):
                loss = F.cross_entropy(net(xb), Y[idx], label_smoothing=c['ls'])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            tl += loss.detach().float() * len(idx)
        bar.set_postfix(loss=f'{tl.item() / n:.4f}', lr=f'{sched.get_last_lr()[0]:.2e}')
    bar.close()
    sec = time.time() - t0
    return net, dict(train_loss=tl.item() / n, sec=sec, sec_per_epoch=sec / c['epochs'],
                     params=sum(p.numel() for p in net.parameters()))


@torch.no_grad()
def get_logits(net, X, bs=1024):
    net.eval()
    return torch.cat([net(X[i:i + bs]).float() for i in range(0, len(X), bs)]).cpu().numpy()


def mf1(y, p, k=11):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm)
    return float((2 * tp / np.maximum(cm.sum(0) + cm.sum(1), 1)).mean())


def scores(y, p, k=11):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm)
    f1 = 2 * tp / np.maximum(cm.sum(0) + cm.sum(1), 1)
    return dict(macro_f1=float(f1.mean()), acc=float(tp.sum() / len(y)), f1=f1.round(4).tolist())


LA = -CFG['la_tau'] * np.log(PRIOR_TR)
print('functions ready; logit adjustment', LA.round(3).tolist())

functions ready; logit adjustment [2.487, 2.859, 1.941, 2.193, 2.515, 3.113, 2.519, 3.098, 2.936, 2.967, 1.391]


In [9]:
# Cell 9: Final fit and export of ensemble members
assert CFG['final_model'] is not None, 'set CFG["final_model"] (and final_la) in Cell 2 after reading Cell 8, rerun Cell 2'
name = CFG['final_model']
c = CFG['models'][name]
if CFG['final_fit'] == 'train+valid':
    Xf, Yf = torch.cat([Xtr, Xva]), torch.cat([Ytr, Yva])
else:
    Xf, Yf = Xtr, Ytr
prior_f = np.bincount(Yf.cpu().numpy(), minlength=11) / len(Yf)
old = glob.glob(os.path.join(CFG['sub_dir'], 'member_*.pt'))
assert len(old) <= len(CFG['final_seeds']), f'stale member files in {CFG["sub_dir"]}: {old}'
for k, seed in enumerate(tqdm(CFG['final_seeds'], desc='final members')):
    net, info = train_model(name, seed, Xf, Yf)
    if CFG['final_la']:
        net.logit_bias.copy_(torch.tensor(-CFG['la_tau'] * np.log(prior_f), dtype=torch.float32, device=dev))
    path = os.path.join(CFG['sub_dir'], f'member_{k}.pt')
    torch.save(dict(arch=c['arch'], size=c['size'], dropout=c['dropout'], num_classes=CFG['num_classes'],
                    state_dict={kk: v.detach().cpu() for kk, v in net.state_dict().items()},
                    train_info=dict(model=name, seed=seed, final_fit=CFG['final_fit'], final_la=CFG['final_la'],
                                    n_train=len(Yf), train_loss=info['train_loss'], sec=info['sec'])), path)
    tqdm.write(f"SAVED {path} | seed {seed} | {info['sec']:.0f}s | params {info['params']:,} | "
               f"train loss {info['train_loss']:.4f} | {os.path.getsize(path) / 2**20:.1f} MB")
    del net
    torch.cuda.empty_cache()
print('members:', sorted(os.listdir(CFG['sub_dir'])))

final members:   0%|          | 0/3 [00:00<?, ?it/s]

r18_in_lr03 s0:   0%|          | 0/15 [00:00<?, ?it/s]

SAVED /content/drive/MyDrive/models/Group 3/member_0.pt | seed 0 | 31s | params 11,182,155 | train loss 0.0267 | 42.7 MB


r18_in_lr03 s1:   0%|          | 0/15 [00:00<?, ?it/s]

SAVED /content/drive/MyDrive/models/Group 3/member_1.pt | seed 1 | 27s | params 11,182,155 | train loss 0.0263 | 42.7 MB


r18_in_lr03 s2:   0%|          | 0/15 [00:00<?, ?it/s]

SAVED /content/drive/MyDrive/models/Group 3/member_2.pt | seed 2 | 27s | params 11,182,155 | train loss 0.0257 | 42.7 MB
members: ['__pycache__', 'member_0.pt', 'member_1.pt', 'member_2.pt', 'predict.py']


In [10]:
# Cell 10: Submission check: predict.py on valid (GPU and CPU)
assert os.path.isfile(os.path.join(CFG['sub_dir'], 'predict.py')), 'upload predict.py into ' + CFG['sub_dir']
for d in tqdm(['cuda', 'cpu'], desc='predict check'):
    os.environ['PREDICT_DEVICE'] = d
    spec = importlib.util.spec_from_file_location(f'predict_{d}', os.path.join(CFG['sub_dir'], 'predict.py'))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    t0 = time.time()
    p = mod.predict(Xva_np.copy())
    dt = time.time() - t0
    assert isinstance(p, np.ndarray) and p.shape == (len(Xva_np),) and p.dtype.kind == 'i' and p.min() >= 0 and p.max() <= 10
    s = scores(Yva_np, p)
    tqdm.write(f"{d}: {dt:.1f}s for {len(p)} images | macroF1 {s['macro_f1']:.4f} acc {s['acc']:.4f} | "
               f"params {sum(sum(q.numel() for q in n.parameters()) for n in mod._load()[1]):,}")
os.environ.pop('PREDICT_DEVICE')
if CFG['final_fit'] == 'train+valid':
    print('NOTE: final members were trained on valid; this valid score is a wiring check, not an estimate')

predict check:   0%|          | 0/2 [00:00<?, ?it/s]

cuda: 0.9s for 2354 images | macroF1 0.9789 acc 0.9907 | params 33,546,465
cpu: 29.4s for 2354 images | macroF1 0.9789 acc 0.9907 | params 33,546,465
NOTE: final members were trained on valid; this valid score is a wiring check, not an estimate
